# Atmospheric Calculations for eVTOL Analysis

This notebook demonstrates how to use the evtol-tools atmosphere module for altitude-dependent calculations in eVTOL aircraft analysis. Topics covered:

- **Altitude Class** - Aviation-specific altitude handling with flight levels and ceiling checks
- **International Standard Atmosphere (ISA)** - Temperature, pressure, density at any altitude
- **Temperature Offsets (ISA+/-)** - Hot day and cold day analysis
- **Altitude Effects on Performance** - How atmospheric conditions affect power requirements
- **Integration with Propulsion** - Using atmosphere in hover power and tip speed calculations
- **Mission Analysis** - Altitude profiles for real-world scenarios
- **Pressure and Density Altitude** - Converting between different altitude references

In [29]:
# Import atmosphere module and supporting quantities
from evtoltools.common import (
    # Atmosphere and Altitude
    Altitude,
    Atmosphere,
    atmosphere_at_altitude,
    sea_level_atmosphere,
    ISA_SEA_LEVEL_TEMPERATURE,
    ISA_SEA_LEVEL_PRESSURE,
    ISA_SEA_LEVEL_DENSITY,
    ISA_SEA_LEVEL_SPEED_OF_SOUND,
    # Quantities
    Length, Temperature, Pressure, Density, Velocity,
    Force, Power, Mass, AngularVelocity,
)
from evtoltools.components import PropulsionSystem, Motor, Propeller
import numpy as np

print("Atmosphere module imported successfully!")
print(f"Altitude class available: {Altitude}")
print(f"Sea level constant: {Altitude.sea_level()}")

Atmosphere module imported successfully!
Altitude class available: <class 'evtoltools.common.atmosphere.altitude.Altitude'>
Sea level constant: 0 meter


---
## Part 1: ISA Sea Level Reference Constants

The International Standard Atmosphere (ISA) defines reference conditions at sea level. These constants are useful for quick reference and validation.

In [30]:
# ISA Sea Level Reference Values
print("ISA SEA LEVEL REFERENCE CONDITIONS")
print("=" * 50)
print(f"Temperature:     {ISA_SEA_LEVEL_TEMPERATURE}")
print(f"                 {ISA_SEA_LEVEL_TEMPERATURE.in_units_of('degC'):.2f} degC")
print(f"                 {ISA_SEA_LEVEL_TEMPERATURE.in_units_of('degF'):.2f} degF")
print()
print(f"Pressure:        {ISA_SEA_LEVEL_PRESSURE}")
print(f"                 {ISA_SEA_LEVEL_PRESSURE.in_units_of('bar'):.5f} bar")
print(f"                 {ISA_SEA_LEVEL_PRESSURE.in_units_of('psi'):.3f} psi")
print()
print(f"Density:         {ISA_SEA_LEVEL_DENSITY}")
print(f"                 {ISA_SEA_LEVEL_DENSITY.in_units_of('lb/ft^3'):.5f} lb/ft^3")
print()
print(f"Speed of Sound:  {ISA_SEA_LEVEL_SPEED_OF_SOUND}")
print(f"                 {ISA_SEA_LEVEL_SPEED_OF_SOUND.in_units_of('km/h'):.1f} km/h")
print(f"                 {ISA_SEA_LEVEL_SPEED_OF_SOUND.in_units_of('knots'):.1f} knots")

ISA SEA LEVEL REFERENCE CONDITIONS
Temperature:     288.15 kelvin
                 15.00 degC
                 59.00 degF

Pressure:        101325 pascal
                 1.01325 bar
                 14.696 psi

Density:         1.225 kilogram / meter ** 3
                 0.07647 lb/ft^3

Speed of Sound:  340.294 meter / second
                 1225.1 km/h
                 661.5 knots


---
## Part 2: The Altitude Class

The `Altitude` class is a semantic wrapper around `Length` specifically designed for aviation. It provides all the same unit conversion capabilities plus aviation-specific features like flight level conversion and troposphere checks.

### 2.1 Creating Altitudes

In [31]:
# Method 1: Direct construction with value and unit
alt_meters = Altitude(5000, 'm')
alt_feet = Altitude(10000, 'ft')
print(f"Direct construction: {alt_meters}, {alt_feet}")

# Method 2: Sea level convenience method
sea_level = Altitude.sea_level()
print(f"Sea level: {sea_level}")

# Method 3: From flight level (FL100 = 10,000 ft)
cruise_fl100 = Altitude.from_flight_level(100)
cruise_fl350 = Altitude.from_flight_level(350)
print(f"FL100: {cruise_fl100}")
print(f"FL350: {cruise_fl350}")

# Method 4: Unit conversion (inherits from Length)
alt_converted = alt_meters.to('ft')
print(f"\n5000m = {alt_converted}")

Direct construction: 5000 meter, 10000 foot
Sea level: 0 meter
FL100: 10000 foot
FL350: 35000 foot

5000m = 16404.199475065616 foot


### 2.2 Flight Level Conversions

Flight levels (FL) are pressure altitudes in hundreds of feet. They're used above the transition altitude for standardized separation.

In [32]:
# Convert flight level to altitude
common_flight_levels = [35, 50, 100, 180, 350]

print("FLIGHT LEVEL TO ALTITUDE CONVERSION")
print("=" * 45)
print(f"{'Flight Level':<15} {'Altitude (ft)':<15} {'Altitude (m)'}")
print("-" * 45)

for fl in common_flight_levels:
    alt = Altitude.from_flight_level(fl)
    print(f"FL{fl:03d}           {alt.in_units_of('ft'):<15.0f} {alt.in_units_of('m'):.0f}")

# Convert altitude to flight level
print("\n\nALTITUDE TO FLIGHT LEVEL CONVERSION")
print("=" * 45)
altitudes_ft = [3500, 10000, 25000, 35000]
for alt_ft in altitudes_ft:
    alt = Altitude(alt_ft, 'ft')
    fl = alt.to_flight_level()
    print(f"{alt_ft:>5} ft  =>  FL{fl:03.0f}")

FLIGHT LEVEL TO ALTITUDE CONVERSION
Flight Level    Altitude (ft)   Altitude (m)
---------------------------------------------
FL035           3500            1067
FL050           5000            1524
FL100           10000           3048
FL180           18000           5486
FL350           35000           10668


ALTITUDE TO FLIGHT LEVEL CONVERSION
 3500 ft  =>  FL035
10000 ft  =>  FL100
25000 ft  =>  FL250
35000 ft  =>  FL350


### 2.3 Aviation-Specific Checks

The Altitude class provides built-in checks for common aviation boundaries.

In [33]:
# Check if altitude is in the troposphere (below ~11km / 36,089 ft)
print("TROPOSPHERE CHECK (below 11km)")
print("=" * 50)
test_altitudes = [
    Altitude(5000, 'ft'),
    Altitude(10000, 'ft'),
    Altitude(35000, 'ft'),
    Altitude(40000, 'ft'),
]

for alt in test_altitudes:
    in_tropo = alt.is_in_troposphere()
    print(f"  {alt.in_units_of('ft'):>6.0f} ft ({alt.in_units_of('m'):>5.0f} m): "
          f"{'In troposphere' if in_tropo else 'Above troposphere'}")

# Check if altitude is below typical eVTOL ceiling (10,000 ft)
print(f"\neVTOL CEILING CHECK (default: {Altitude.EVTOL_TYPICAL_CEILING_FT} ft)")
print("=" * 50)

evtol_altitudes = [
    Altitude(1500, 'ft'),
    Altitude(5000, 'ft'),
    Altitude(10000, 'ft'),
    Altitude(12000, 'ft'),
]

for alt in evtol_altitudes:
    below_ceiling = alt.is_below_evtol_ceiling()
    custom_ceiling = alt.is_below_evtol_ceiling(ceiling_ft=5000)
    print(f"  {alt.in_units_of('ft'):>6.0f} ft: "
          f"Below 10k: {str(below_ceiling):<6} | Below 5k: {custom_ceiling}")

# AGL reference string
print("\nAGL REFERENCE STRINGS")
print("=" * 50)
for alt in [Altitude(1500, 'ft'), Altitude(500, 'm')]:
    print(f"  {alt} => {alt.agl_reference}")

TROPOSPHERE CHECK (below 11km)
    5000 ft ( 1524 m): In troposphere
   10000 ft ( 3048 m): In troposphere
   35000 ft (10668 m): In troposphere
   40000 ft (12192 m): Above troposphere

eVTOL CEILING CHECK (default: 10000 ft)
    1500 ft: Below 10k: True   | Below 5k: True
    5000 ft: Below 10k: True   | Below 5k: True
   10000 ft: Below 10k: True   | Below 5k: False
   12000 ft: Below 10k: False  | Below 5k: False

AGL REFERENCE STRINGS
  1500 foot => 1500 ft AGL
  500 meter => 1640 ft AGL


---
## Part 3: Creating Atmosphere Objects

The `Atmosphere` class provides atmospheric properties at any altitude. It accepts both `Altitude` and `Length` objects for flexibility.

### 3.1 Basic Construction with Altitude

In [34]:
# Method 1: Using Atmosphere class with Altitude (preferred)
atm_5000m = Atmosphere(Altitude(5000, 'm'))
print(f"Atmosphere at 5000m: {atm_5000m}")

# Method 2: Using flight level
atm_fl100 = Atmosphere(Altitude.from_flight_level(100))
print(f"Atmosphere at FL100: {atm_fl100}")

# Method 3: Using Length (still supported for backwards compatibility)
atm_10000ft = Atmosphere(Length(10000, 'ft'))
print(f"Atmosphere at 10000ft: {atm_10000ft}")

# Method 4: Using convenience functions
atm_cruise = atmosphere_at_altitude(Altitude(3000, 'ft'))
print(f"Atmosphere at 3000ft: {atm_cruise}")

# Method 5: Sea level convenience function
atm_sl = sea_level_atmosphere()
print(f"Sea level atmosphere: {atm_sl}")

# The altitude property now returns an Altitude object
print(f"\nAltitude type from Atmosphere: {type(atm_5000m.altitude).__name__}")
print(f"Flight level: FL{atm_fl100.altitude.to_flight_level():.0f}")

Atmosphere at 5000m: Atmosphere at 5000 meter: T=255.67554322180348 kelvin, P=54048.26223756018 pascal, rho=0.7364286133691456 kilogram / meter ** 3
Atmosphere at FL100: Atmosphere at 10000 foot: T=268.3474950852336 kelvin, P=69694.60186793635 pascal, rho=0.9047731467868786 kilogram / meter ** 3
Atmosphere at 10000ft: Atmosphere at 10000 foot: T=268.3474950852336 kelvin, P=69694.60186793635 pascal, rho=0.9047731467868786 kilogram / meter ** 3
Atmosphere at 3000ft: Atmosphere at 3000 foot: T=282.2072548444555 kelvin, P=90813.10739559718 pascal, rho=1.1210331849858202 kilogram / meter ** 3
Sea level atmosphere: Atmosphere at 0 meter: T=288.15 kelvin, P=101325.0 pascal, rho=1.225000018124288 kilogram / meter ** 3

Altitude type from Atmosphere: Altitude
Flight level: FL100


### 3.2 Accessing Atmospheric Properties

In [35]:
# Create atmosphere at typical eVTOL cruise altitude using Altitude class
cruise_alt = Altitude(1500, 'ft')
atm = Atmosphere(cruise_alt)

print(f"ATMOSPHERIC PROPERTIES AT {cruise_alt}")
print("=" * 50)

# Temperature
print(f"Temperature:     {atm.temperature}")
print(f"                 {atm.temperature.in_units_of('degC'):.2f} degC")

# Pressure
print(f"\nPressure:        {atm.pressure}")
print(f"                 {atm.pressure.in_units_of('kPa'):.3f} kPa")

# Density
print(f"\nDensity:         {atm.density}")
print(f"                 {atm.density.in_units_of('kg/m^3'):.4f} kg/m^3")

# Speed of Sound
print(f"\nSpeed of Sound:  {atm.speed_of_sound}")
print(f"                 {atm.speed_of_sound.in_units_of('m/s'):.2f} m/s")

# Viscosity (for Reynolds number calculations)
print(f"\nKinematic Viscosity: {atm.kinematic_viscosity:.2e} m^2/s")
print(f"Dynamic Viscosity:   {atm.dynamic_viscosity:.2e} Pa*s")

# Aviation-specific checks via Altitude
print(f"\nAviation Checks:")
print(f"  In troposphere:     {cruise_alt.is_in_troposphere()}")
print(f"  Below eVTOL ceiling: {cruise_alt.is_below_evtol_ceiling()}")
print(f"  Flight level:       FL{cruise_alt.to_flight_level():.0f}")

ATMOSPHERIC PROPERTIES AT 1500 foot
Temperature:     285.1784137264836 kelvin
                 12.03 degC

Pressure:        95952.1638643397 pascal
                 95.952 kPa

Density:         1.1721312124218237 kilogram / meter ** 3
                 1.1721 kg/m^3

Speed of Sound:  338.53477660519354 meter / second
                 338.53 m/s

Kinematic Viscosity: 1.51e-05 m^2/s
Dynamic Viscosity:   1.78e-05 Pa*s

Aviation Checks:
  In troposphere:     True
  Below eVTOL ceiling: True
  Flight level:       FL15


---
## Part 4: Temperature Offsets (ISA+/- Analysis)

Real-world conditions often differ from standard ISA. Use temperature offsets to model hot day (ISA+) or cold day (ISA-) scenarios. This is critical for performance certification.

### 4.1 Creating Atmosphere with Temperature Offset

In [36]:
# Use Altitude for clearer semantics
altitude = Altitude(2000, 'ft')

# Standard day (no offset)
atm_std = Atmosphere(altitude)

# Hot day: ISA + 20K (common certification condition)
atm_hot = Atmosphere(altitude, temperature_offset=Temperature(20, 'K'))

# Cold day: ISA - 15K
atm_cold = Atmosphere(altitude, temperature_offset=Temperature(-15, 'K'))

print(f"TEMPERATURE OFFSET COMPARISON AT {altitude}")
print("=" * 60)
print(f"{'Condition':<15} {'Temperature':<18} {'Density':<20} {'Speed of Sound'}")
print("-" * 60)
print(f"{'ISA-15 (Cold)':<15} {atm_cold.temperature.in_units_of('degC'):>8.1f} degC    "
      f"{atm_cold.density.in_units_of('kg/m^3'):>8.4f} kg/m^3    "
      f"{atm_cold.speed_of_sound.in_units_of('m/s'):>6.1f} m/s")
print(f"{'ISA (Standard)':<15} {atm_std.temperature.in_units_of('degC'):>8.1f} degC    "
      f"{atm_std.density.in_units_of('kg/m^3'):>8.4f} kg/m^3    "
      f"{atm_std.speed_of_sound.in_units_of('m/s'):>6.1f} m/s")
print(f"{'ISA+20 (Hot)':<15} {atm_hot.temperature.in_units_of('degC'):>8.1f} degC    "
      f"{atm_hot.density.in_units_of('kg/m^3'):>8.4f} kg/m^3    "
      f"{atm_hot.speed_of_sound.in_units_of('m/s'):>6.1f} m/s")

TEMPERATURE OFFSET COMPARISON AT 2000 foot
Condition       Temperature        Density              Speed of Sound
------------------------------------------------------------
ISA-15 (Cold)       -4.0 degC      1.2193 kg/m^3     328.9 m/s
ISA (Standard)      11.0 degC      1.1549 kg/m^3     337.9 m/s
ISA+20 (Hot)        31.0 degC      1.0790 kg/m^3     349.6 m/s


### 4.2 Key Point: Pressure is Unaffected by Temperature Offset

Temperature offset only affects temperature-dependent properties. Pressure is based purely on geometric altitude.

In [37]:
# Pressure remains the same regardless of temperature offset
print("PRESSURE COMPARISON (should be identical):")
print(f"  ISA-15:    {atm_cold.pressure.in_units_of('Pa'):.2f} Pa")
print(f"  ISA:       {atm_std.pressure.in_units_of('Pa'):.2f} Pa")
print(f"  ISA+20:    {atm_hot.pressure.in_units_of('Pa'):.2f} Pa")

# Verify they're equal
print(f"\nAll equal: {atm_cold.pressure == atm_std.pressure == atm_hot.pressure}")

PRESSURE COMPARISON (should be identical):
  ISA-15:    94213.56 Pa
  ISA:       94213.56 Pa
  ISA+20:    94213.56 Pa

All equal: True


### 4.3 Accessing ISA Properties Separately

When using temperature offsets, you can still access the underlying ISA values.

In [38]:
atm_hot = Atmosphere(Altitude(5000, 'm'), temperature_offset=Temperature(25, 'K'))

print("HOT DAY (ISA+25) AT 5000m")
print("=" * 50)
print(f"Actual temperature:    {atm_hot.temperature.in_units_of('degC'):.1f} degC")
print(f"ISA temperature:       {atm_hot.isa_temperature.in_units_of('degC'):.1f} degC")
print(f"Temperature offset:    {atm_hot.temperature_offset}")
print()
print(f"Actual density:        {atm_hot.density.in_units_of('kg/m^3'):.4f} kg/m^3")
print(f"ISA density:           {atm_hot.isa_density.in_units_of('kg/m^3'):.4f} kg/m^3")
print(f"Density reduction:     {(1 - atm_hot.density.in_units_of('kg/m^3') / atm_hot.isa_density.in_units_of('kg/m^3')) * 100:.1f}%")
print()
print(f"Actual speed of sound: {atm_hot.speed_of_sound.in_units_of('m/s'):.1f} m/s")
print(f"ISA speed of sound:    {atm_hot.isa_speed_of_sound.in_units_of('m/s'):.1f} m/s")

HOT DAY (ISA+25) AT 5000m
Actual temperature:    7.5 degC
ISA temperature:       -17.5 degC
Temperature offset:    25 kelvin

Actual density:        0.6708 kg/m^3
ISA density:           0.7364 kg/m^3
Density reduction:     8.9%

Actual speed of sound: 335.9 m/s
ISA speed of sound:    320.5 m/s


---
## Part 5: Altitude Effects on Atmospheric Properties

Understanding how properties change with altitude is essential for mission planning.

### 5.1 Property Variation with Altitude

In [39]:
# Analyze properties at multiple altitudes using Altitude class
altitudes_ft = [0, 1000, 2000, 3000, 5000, 10000, 15000]

print("ATMOSPHERIC PROPERTIES VS ALTITUDE")
print("=" * 80)
print(f"{'Altitude':<12} {'Temperature':<14} {'Pressure':<14} {'Density':<16} {'Speed of Sound'}")
print(f"{'(ft)':<12} {'(degC)':<14} {'(kPa)':<14} {'(kg/m^3)':<16} {'(m/s)'}")
print("-" * 80)

for alt_ft in altitudes_ft:
    alt = Altitude(alt_ft, 'ft')
    atm = Atmosphere(alt)
    print(f"{alt_ft:<12} {atm.temperature.in_units_of('degC'):<14.1f} "
          f"{atm.pressure.in_units_of('kPa'):<14.2f} "
          f"{atm.density.in_units_of('kg/m^3'):<16.4f} "
          f"{atm.speed_of_sound.in_units_of('m/s'):.1f}")

ATMOSPHERIC PROPERTIES VS ALTITUDE
Altitude     Temperature    Pressure       Density          Speed of Sound
(ft)         (degC)         (kPa)          (kg/m^3)         (m/s)
--------------------------------------------------------------------------------
0            15.0           101.33         1.2250           340.3
1000         13.0           97.72          1.1896           339.1
2000         11.0           94.21          1.1549           337.9
3000         9.1            90.81          1.1210           336.8
5000         5.1            84.31          1.0556           334.4
10000        -4.8           69.69          0.9048           328.4
15000        -14.7          57.21          0.7711           322.3


### 5.2 Using NumPy Arrays for Batch Calculations

In [40]:
# Create atmosphere with array of altitudes (Altitude also supports arrays)
altitudes = np.linspace(0, 5000, 11)  # 0 to 5000m in 500m increments
atm_profile = Atmosphere(Altitude(altitudes, 'm'))

# All properties are now arrays
print("ARRAY-BASED ALTITUDE PROFILE")
print("=" * 60)
print(f"Altitudes (m):      {altitudes}")
print(f"\nTemperatures (K):   {atm_profile.temperature.magnitude}")
print(f"\nDensities (kg/m^3): {atm_profile.density.magnitude}")

# Calculate density ratio relative to sea level
density_ratio = atm_profile.density.magnitude / ISA_SEA_LEVEL_DENSITY.magnitude
print(f"\nDensity ratio:      {np.round(density_ratio, 3)}")

ARRAY-BASED ALTITUDE PROFILE
Altitudes (m):      [   0.  500. 1000. 1500. 2000. 2500. 3000. 3500. 4000. 4500. 5000.]

Temperatures (K):   [288.15       284.90025561 281.65102237 278.40230016 275.15408884
 271.90638832 268.65919845 265.41251913 262.16635023 258.92069164
 255.67554322]

Densities (kg/m^3): [1.22500002 1.16727328 1.11165967 1.05810446 1.00655375 0.95695447
 0.90925435 0.86340194 0.8193466  0.7770385  0.73642861]

Density ratio:      [1.    0.953 0.907 0.864 0.822 0.781 0.742 0.705 0.669 0.634 0.601]


---
## Part 6: Pressure and Density Altitude

Pilots and engineers often work with **pressure altitude** (altitude corresponding to a given pressure) and **density altitude** (altitude corresponding to a given density). The `Altitude` class provides direct methods for these conversions.

### 6.1 Calculating Pressure Altitude

In [41]:
# Given a pressure reading, find the corresponding ISA altitude
measured_pressure = Pressure(95000, 'Pa')  # Lower than sea level

# Method 1: Use Altitude.from_pressure() directly (returns Altitude)
pressure_alt = Altitude.from_pressure(measured_pressure)
print(f"Measured pressure: {measured_pressure}")
print(f"Pressure altitude: {pressure_alt.to('ft')}")
print(f"                   {pressure_alt.to('m')}")
print(f"Flight level:      FL{pressure_alt.to_flight_level():.0f}")

# Method 2: Get full atmosphere at that pressure altitude
atm_from_pressure = Atmosphere.from_pressure_altitude(measured_pressure)
print(f"\nFull atmosphere at pressure altitude:")
print(f"  Temperature: {atm_from_pressure.temperature.in_units_of('degC'):.1f} degC")
print(f"  Density:     {atm_from_pressure.density.in_units_of('kg/m^3'):.4f} kg/m^3")

Measured pressure: 95000 pascal
Pressure altitude: 1772.9331189253198 foot
                   540.3900146484375 meter
Flight level:      FL18

Full atmosphere at pressure altitude:
  Temperature: 11.5 degC
  Density:     1.1627 kg/m^3


### 6.2 Calculating Density Altitude

In [42]:
# Given a density, find the corresponding ISA altitude
# This is crucial for performance - aircraft "feels" like it's at this altitude

measured_density = Density(1.0, 'kg/m^3')  # Lower than sea level ISA

# Use Altitude.from_density() directly (returns Altitude)
density_alt = Altitude.from_density(measured_density)
print(f"Measured density:  {measured_density}")
print(f"Density altitude:  {density_alt.to('ft')}")
print(f"                   {density_alt.to('m')}")
print(f"Flight level:      FL{density_alt.to_flight_level():.0f}")

# Aviation checks
print(f"\nAviation checks:")
print(f"  In troposphere:      {density_alt.is_in_troposphere()}")
print(f"  Below eVTOL ceiling: {density_alt.is_below_evtol_ceiling()}")

# Compare to sea level
print(f"\nSea level density: {ISA_SEA_LEVEL_DENSITY}")
print(f"Density reduction: {(1 - measured_density.magnitude / ISA_SEA_LEVEL_DENSITY.magnitude) * 100:.1f}%")

Measured density:  1.0 kilogram / meter ** 3
Density altitude:  6774.591961557784 foot
                   2064.8956298828125 meter
Flight level:      FL68

Aviation checks:
  In troposphere:      True
  Below eVTOL ceiling: True

Sea level density: 1.225 kilogram / meter ** 3
Density reduction: 18.4%


### 6.3 Practical Example: Hot Day Density Altitude

In [43]:
# A helipad at 2000 ft on a hot day (ISA+20)
geometric_altitude = Altitude(2000, 'ft')
atm_hot_day = Atmosphere(geometric_altitude, temperature_offset=Temperature(20, 'K'))

# Find the density altitude using Altitude.from_density()
actual_density = atm_hot_day.density
density_altitude = Altitude.from_density(actual_density)

print("HOT DAY DENSITY ALTITUDE CALCULATION")
print("=" * 50)
print(f"Geometric altitude:  {geometric_altitude}")
print(f"Temperature offset:  ISA+20K")
print(f"Actual temperature:  {atm_hot_day.temperature.in_units_of('degC'):.1f} degC")
print(f"Actual density:      {actual_density.in_units_of('kg/m^3'):.4f} kg/m^3")
print(f"\nDensity altitude:    {density_altitude.to('ft')}")
print(f"Flight level equiv:  FL{density_altitude.to_flight_level():.0f}")
print(f"Effective increase:  {density_altitude.in_units_of('ft') - geometric_altitude.in_units_of('ft'):.0f} ft")
print(f"\n=> Aircraft performs as if at {density_altitude.in_units_of('ft'):.0f} ft on a standard day!")

# Check if we're still in safe operating range
print(f"\nSafety check:")
print(f"  Below eVTOL ceiling (10k ft): {density_altitude.is_below_evtol_ceiling()}")

HOT DAY DENSITY ALTITUDE CALCULATION
Geometric altitude:  2000 foot
Temperature offset:  ISA+20K
Actual temperature:  31.0 degC
Actual density:      1.0790 kg/m^3

Density altitude:    4274.513464900139 foot
Flight level equiv:  FL43
Effective increase:  2275 ft

=> Aircraft performs as if at 4275 ft on a standard day!

Safety check:
  Below eVTOL ceiling (10k ft): True


---
## Part 7: Integration with Propulsion Calculations

The atmosphere module integrates directly with propulsion components for altitude-dependent performance analysis. You can pass either `Altitude` or `Atmosphere` objects.

### 7.1 Hover Power vs Altitude

In [44]:
# Create a propulsion system
motor = Motor(max_power=Power(50, 'kW'), efficiency=0.92)
propeller = Propeller(diameter=Length(1.8, 'm'), efficiency_hover=0.72)
propulsion = PropulsionSystem(
    motors=[motor],
    propellers=[propeller],
    num_units=4,
    installation_efficiency=0.95
)

# Aircraft weight
mtow = Mass(800, 'kg')
weight = Force(mtow.magnitude * 9.81, 'N')

print(f"PROPULSION SYSTEM: {propulsion.num_units} rotors, {propeller.diameter} diameter")
print(f"AIRCRAFT WEIGHT: {weight.in_units_of('N'):.0f} N ({mtow})")
print()

PROPULSION SYSTEM: 4 rotors, 1.8 meter diameter
AIRCRAFT WEIGHT: 7848 N (800 kilogram)



In [45]:
# Compare hover power at different altitudes
altitudes_ft = [0, 1000, 2000, 3000, 5000]

print("HOVER POWER VS ALTITUDE")
print("=" * 70)
print(f"{'Altitude':<12} {'Density':<16} {'Ideal Power':<14} {'Shaft Power':<14} {'Elec Power'}")
print(f"{'(ft)':<12} {'(kg/m^3)':<16} {'(kW)':<14} {'(kW)':<14} {'(kW)'}")
print("-" * 70)

for alt_ft in altitudes_ft:
    # Create Altitude and Atmosphere objects
    alt = Altitude(alt_ft, 'ft')
    atm = Atmosphere(alt)
    
    # Can pass Altitude, Atmosphere, or Length to atmosphere parameter
    ideal = propulsion.hover_power_ideal(weight, atmosphere=alt)
    shaft = propulsion.hover_shaft_power(weight, atmosphere=atm)
    elec = propulsion.hover_electrical_power(weight, atmosphere=atm)
    
    print(f"{alt_ft:<12} {atm.density.in_units_of('kg/m^3'):<16.4f} "
          f"{ideal.in_units_of('kW'):<14.2f} {shaft.in_units_of('kW'):<14.2f} "
          f"{elec.in_units_of('kW'):.2f}")

# Calculate power increase from sea level to 5000 ft
power_sl = propulsion.hover_electrical_power(weight, atmosphere=Altitude.sea_level())
power_5k = propulsion.hover_electrical_power(weight, atmosphere=Altitude(5000, 'ft'))
increase = (power_5k.in_units_of('kW') / power_sl.in_units_of('kW') - 1) * 100
print(f"\nPower increase from sea level to 5000 ft: {increase:.1f}%")

HOVER POWER VS ALTITUDE
Altitude     Density          Ideal Power    Shaft Power    Elec Power
(ft)         (kg/m^3)         (kW)           (kW)           (kW)
----------------------------------------------------------------------
0            1.2250           139.22         203.54         221.24
1000         1.1896           141.28         206.55         224.51
2000         1.1549           143.38         209.63         227.86
3000         1.1210           145.53         212.77         231.27
5000         1.0556           149.98         219.27         238.33

Power increase from sea level to 5000 ft: 7.7%


### 7.2 Hot Day Performance Impact

In [46]:
# Compare standard day vs hot day at a typical vertiport altitude
vertiport_alt = Altitude(1500, 'ft')

atm_std = Atmosphere(vertiport_alt)
atm_hot = Atmosphere(vertiport_alt, temperature_offset=Temperature(20, 'K'))
atm_extreme = Atmosphere(vertiport_alt, temperature_offset=Temperature(35, 'K'))

print(f"HOT DAY PERFORMANCE IMPACT AT {vertiport_alt}")
print("=" * 60)

conditions = [
    ("Standard (ISA)", atm_std),
    ("Hot (ISA+20K)", atm_hot),
    ("Extreme (ISA+35K)", atm_extreme),
]

print(f"{'Condition':<20} {'Temp (C)':<12} {'Density':<14} {'Hover Power (kW)'}")
print("-" * 60)

power_std = None
for name, atm in conditions:
    power = propulsion.hover_electrical_power(weight, atmosphere=atm)
    if power_std is None:
        power_std = power.in_units_of('kW')
        increase_str = "(baseline)"
    else:
        increase = (power.in_units_of('kW') / power_std - 1) * 100
        increase_str = f"(+{increase:.1f}%)"
    
    print(f"{name:<20} {atm.temperature.in_units_of('degC'):<12.1f} "
          f"{atm.density.in_units_of('kg/m^3'):<14.4f} "
          f"{power.in_units_of('kW'):.2f} {increase_str}")

HOT DAY PERFORMANCE IMPACT AT 1500 foot
Condition            Temp (C)     Density        Hover Power (kW)
------------------------------------------------------------
Standard (ISA)       12.0         1.1721         226.17 (baseline)
Hot (ISA+20K)        32.0         1.0953         233.97 (+3.4%)
Extreme (ISA+35K)    47.0         1.0440         239.65 (+6.0%)


### 7.3 Induced Velocity vs Altitude

In [47]:
# Induced velocity increases with altitude due to lower air density
print("INDUCED VELOCITY VS ALTITUDE")
print("=" * 50)
print(f"{'Altitude (ft)':<15} {'Density (kg/m^3)':<18} {'Induced Vel (m/s)'}")
print("-" * 50)

for alt_ft in [0, 2000, 4000, 6000, 8000]:
    alt = Altitude(alt_ft, 'ft')
    atm = Atmosphere(alt)
    vi = propulsion.induced_velocity(weight, atmosphere=atm)
    print(f"{alt_ft:<15} {atm.density.in_units_of('kg/m^3'):<18.4f} {vi.in_units_of('m/s'):.2f}")

print("\n=> Higher induced velocity means more downwash and noise at altitude!")

INDUCED VELOCITY VS ALTITUDE
Altitude (ft)   Density (kg/m^3)   Induced Vel (m/s)
--------------------------------------------------
0               1.2250             17.74
2000            1.1549             18.27
4000            1.0879             18.82
6000            1.0240             19.40
8000            0.9630             20.01

=> Higher induced velocity means more downwash and noise at altitude!


### 7.4 Tip Speed and Mach Number Limits

In [48]:
# Maximum tip speed decreases with altitude due to lower speed of sound
print("MAXIMUM TIP SPEED VS ALTITUDE")
print("=" * 60)
print(f"{'Altitude (ft)':<15} {'Speed of Sound':<18} {'Max Tip Speed':<16} {'Max RPM'}")
print("-" * 60)

for alt_ft in [0, 5000, 10000, 15000]:
    alt = Altitude(alt_ft, 'ft')
    atm = Atmosphere(alt)
    max_tip = propeller.max_tip_speed(atmosphere=atm)
    max_omega = propeller.max_angular_velocity(atmosphere=atm)
    
    print(f"{alt_ft:<15} {atm.speed_of_sound.in_units_of('m/s'):<18.1f} "
          f"{max_tip.in_units_of('m/s'):<16.1f} {max_omega.in_units_of('rpm'):.0f}")

print(f"\nPropeller tip Mach limit: {propeller.tip_mach_limit}")

MAXIMUM TIP SPEED VS ALTITUDE
Altitude (ft)   Speed of Sound     Max Tip Speed    Max RPM
------------------------------------------------------------
0               340.3              289.2            3069
5000            334.4              284.2            3016
10000           328.4              279.1            2962
15000           322.3              273.9            2907

Propeller tip Mach limit: 0.85


In [49]:
# Calculate Mach number at a given RPM and altitude
operating_rpm = AngularVelocity(2500, 'rpm')

print(f"TIP MACH NUMBER AT {operating_rpm} ACROSS ALTITUDES")
print("=" * 50)

for alt_ft in [0, 5000, 10000]:
    alt = Altitude(alt_ft, 'ft')
    atm = Atmosphere(alt)
    mach = propeller.tip_mach_number(operating_rpm, atmosphere=atm)
    status = "OK" if mach < propeller.tip_mach_limit else "EXCEEDS LIMIT!"
    print(f"  {alt_ft:>5} ft: Mach {mach:.3f} [{status}]")

TIP MACH NUMBER AT 2500 revolutions_per_minute ACROSS ALTITUDES
      0 ft: Mach 0.692 [OK]
   5000 ft: Mach 0.705 [OK]
  10000 ft: Mach 0.717 [OK]


---
## Part 8: Mission Analysis with Altitude Profiles

Realistic mission analysis requires considering atmospheric conditions at each phase of flight.

### 8.1 Simple Mission Profile

In [50]:
# Define mission phases with Altitude objects
mission_phases = [
    {"name": "Takeoff Hover", "altitude": Altitude(50, 'ft'), "duration_min": 1},
    {"name": "Climb", "altitude": Altitude(1000, 'ft'), "duration_min": 2},
    {"name": "Cruise", "altitude": Altitude(2000, 'ft'), "duration_min": 15},
    {"name": "Descent", "altitude": Altitude(1000, 'ft'), "duration_min": 2},
    {"name": "Landing Hover", "altitude": Altitude(50, 'ft'), "duration_min": 1},
]

print("MISSION ENERGY ANALYSIS (Standard Day)")
print("=" * 75)
print(f"{'Phase':<18} {'Alt (ft)':<10} {'Duration':<10} {'Power (kW)':<12} {'Energy (kWh)'}")
print("-" * 75)

total_energy_wh = 0
for phase in mission_phases:
    alt = phase["altitude"]
    atm = Atmosphere(alt)
    power = propulsion.hover_electrical_power(weight, atmosphere=atm)
    energy_wh = power.in_units_of('W') * phase["duration_min"] / 60
    total_energy_wh += energy_wh
    
    print(f"{phase['name']:<18} {alt.in_units_of('ft'):<10.0f} "
          f"{phase['duration_min']:<10} min {power.in_units_of('kW'):<12.2f} "
          f"{energy_wh/1000:.3f}")

print("-" * 75)
print(f"{'TOTAL':<18} {'':<10} {sum(p['duration_min'] for p in mission_phases):<10} min "
      f"{'':<12} {total_energy_wh/1000:.3f}")

MISSION ENERGY ANALYSIS (Standard Day)
Phase              Alt (ft)   Duration   Power (kW)   Energy (kWh)
---------------------------------------------------------------------------
Takeoff Hover      50         1          min 221.40       3.690
Climb              1000       2          min 224.51       7.484
Cruise             2000       15         min 227.86       56.964
Descent            1000       2          min 224.51       7.484
Landing Hover      50         1          min 221.40       3.690
---------------------------------------------------------------------------
TOTAL                         21         min              79.311


### 8.2 Hot Day Mission Comparison

In [51]:
# Compare standard day vs hot day energy consumption
def calculate_mission_energy(temp_offset_k=0):
    """Calculate total mission energy for given temperature offset."""
    total_wh = 0
    for phase in mission_phases:
        alt = phase["altitude"]
        if temp_offset_k == 0:
            atm = Atmosphere(alt)
        else:
            atm = Atmosphere(alt, temperature_offset=Temperature(temp_offset_k, 'K'))
        power = propulsion.hover_electrical_power(weight, atmosphere=atm)
        total_wh += power.in_units_of('W') * phase["duration_min"] / 60
    return total_wh / 1000  # Return kWh

energy_std = calculate_mission_energy(0)
energy_hot = calculate_mission_energy(20)
energy_extreme = calculate_mission_energy(35)

print("MISSION ENERGY: TEMPERATURE COMPARISON")
print("=" * 50)
print(f"Standard day (ISA):     {energy_std:.3f} kWh")
print(f"Hot day (ISA+20K):      {energy_hot:.3f} kWh (+{(energy_hot/energy_std-1)*100:.1f}%)")
print(f"Extreme (ISA+35K):      {energy_extreme:.3f} kWh (+{(energy_extreme/energy_std-1)*100:.1f}%)")
print()
print("=> Battery sizing must account for hot day conditions!")

MISSION ENERGY: TEMPERATURE COMPARISON
Standard day (ISA):     79.311 kWh
Hot day (ISA+20K):      82.048 kWh (+3.5%)
Extreme (ISA+35K):      84.042 kWh (+6.0%)

=> Battery sizing must account for hot day conditions!


### 8.3 High Altitude Operations

In [52]:
# Compare operations at sea level vs high altitude vertiport (e.g., Denver)
print("VERTIPORT LOCATION COMPARISON")
print("=" * 75)

locations = [
    ("Los Angeles (sea level)", Altitude.sea_level()),
    ("Dallas (430 ft)", Altitude(430, 'ft')),
    ("Denver (5280 ft)", Altitude(5280, 'ft')),
    ("Mexico City (7380 ft)", Altitude(7380, 'ft')),
]

print(f"{'Location':<30} {'Alt (ft)':<12} {'Density':<14} {'Hover Power':<15} {'Below Ceiling'}")
print("-" * 75)

base_power = None
for name, alt in locations:
    atm = Atmosphere(alt)
    power = propulsion.hover_electrical_power(weight, atmosphere=atm)
    below_ceiling = alt.is_below_evtol_ceiling()
    
    if base_power is None:
        base_power = power.in_units_of('kW')
        pct = ""
    else:
        pct = f" (+{(power.in_units_of('kW')/base_power - 1)*100:.1f}%)"
    
    print(f"{name:<30} {alt.in_units_of('ft'):<12.0f} {atm.density.in_units_of('kg/m^3'):<14.4f} "
          f"{power.in_units_of('kW'):.2f} kW{pct:<8} {below_ceiling}")

VERTIPORT LOCATION COMPARISON
Location                       Alt (ft)     Density        Hover Power     Below Ceiling
---------------------------------------------------------------------------
Los Angeles (sea level)        0            1.2250         221.24 kW         True
Dallas (430 ft)                430          1.2097         222.64 kW (+0.6%) True
Denver (5280 ft)               5280         1.0467         239.35 kW (+8.2%) True
Mexico City (7380 ft)          7380         0.9816         247.16 kW (+11.7%) True


---
## Part 9: Advanced Topics

### 9.1 Troposphere vs Stratosphere

In [53]:
# In the troposphere (0-11km), temperature decreases with altitude
# In the stratosphere (11-20km), temperature is roughly constant
# The Altitude class provides is_in_troposphere() for convenience

print("TEMPERATURE PROFILE: TROPOSPHERE TO STRATOSPHERE")
print("=" * 70)

altitudes_km = [0, 2, 4, 6, 8, 10, 11, 12, 15, 20]

print(f"{'Altitude (km)':<15} {'Temperature (K)':<18} {'Temperature (C)':<18} {'Region'}")
print("-" * 70)

for alt_km in altitudes_km:
    alt = Altitude(alt_km * 1000, 'm')
    atm = Atmosphere(alt)
    region = "Troposphere" if alt.is_in_troposphere() else "Strato/Tropopause"
    print(f"{alt_km:<15} {atm.temperature.in_units_of('K'):<18.2f} "
          f"{atm.temperature.in_units_of('degC'):<18.2f} {region}")

print(f"\nTropopause boundary: {Altitude.TROPOPAUSE_M} m / {Altitude.TROPOPAUSE_FT} ft")
print("=> Temperature stops decreasing above the tropopause")

TEMPERATURE PROFILE: TROPOSPHERE TO STRATOSPHERE
Altitude (km)   Temperature (K)    Temperature (C)    Region
----------------------------------------------------------------------
0               288.15             15.00              Troposphere
2               275.15             2.00               Troposphere
4               262.17             -10.98             Troposphere
6               249.19             -23.96             Troposphere
8               236.22             -36.93             Troposphere
10              223.25             -49.90             Troposphere
11              216.77             -56.38             Strato/Tropopause
12              216.65             -56.50             Strato/Tropopause
15              216.65             -56.50             Strato/Tropopause
20              216.65             -56.50             Strato/Tropopause

Tropopause boundary: 11000 m / 36089 ft
=> Temperature stops decreasing above the tropopause


### 9.2 Reynolds Number Calculation

In [54]:
# Reynolds number is important for aerodynamic analysis
# Re = (rho * V * L) / mu = (V * L) / nu

blade_chord = Length(0.15, 'm')  # Blade chord length
tip_speed = Velocity(200, 'm/s')  # Blade tip speed

print("REYNOLDS NUMBER VS ALTITUDE")
print("=" * 55)
print(f"Blade chord: {blade_chord}, Tip speed: {tip_speed}")
print()
print(f"{'Altitude (ft)':<15} {'Kin. Visc (m^2/s)':<20} {'Reynolds Number'}")
print("-" * 55)

for alt_ft in [0, 5000, 10000, 20000]:
    alt = Altitude(alt_ft, 'ft')
    atm = Atmosphere(alt)
    Re = (tip_speed.in_units_of('m/s') * blade_chord.in_units_of('m')) / atm.kinematic_viscosity
    print(f"{alt_ft:<15} {atm.kinematic_viscosity:<20.2e} {Re:.2e}")

print("\n=> Reynolds number decreases with altitude (affects blade aerodynamics)")

REYNOLDS NUMBER VS ALTITUDE
Blade chord: 0.15 meter, Tip speed: 200 meter / second

Altitude (ft)   Kin. Visc (m^2/s)    Reynolds Number
-------------------------------------------------------
0               1.46e-05             2.05e+06
5000            1.65e-05             1.82e+06
10000           1.87e-05             1.60e+06
20000           2.44e-05             1.23e+06

=> Reynolds number decreases with altitude (affects blade aerodynamics)


### 9.3 Backwards Compatibility

In [55]:
# All propulsion methods still work without atmosphere parameter
# They default to sea level ISA conditions

# Old way (still works)
power_old = propulsion.hover_power_ideal(weight)
power_with_density = propulsion.hover_power_ideal(weight, air_density=Density(1.225, 'kg/m^3'))

# New way with Altitude
power_altitude = propulsion.hover_power_ideal(weight, atmosphere=Altitude.sea_level())

# Also works with Length (backwards compatible)
power_length = propulsion.hover_power_ideal(weight, atmosphere=Altitude(0, 'm'))

print("BACKWARDS COMPATIBILITY CHECK")
print("=" * 55)
print(f"Old method (no params):       {power_old.in_units_of('kW'):.4f} kW")
print(f"Old method (with density):    {power_with_density.in_units_of('kW'):.4f} kW")
print(f"New method (Altitude):        {power_altitude.in_units_of('kW'):.4f} kW")
print(f"New method (Length):          {power_length.in_units_of('kW'):.4f} kW")

# Using direct quantity comparison (demonstrates abs() and subtraction on quantities)
print(f"\nAll methods give same result: {abs(power_old - power_altitude).in_units_of('kW') < 0.001}")

BACKWARDS COMPATIBILITY CHECK
Old method (no params):       139.2220 kW
Old method (with density):    139.2220 kW
New method (Altitude):        139.2220 kW
New method (Length):          139.2220 kW

All methods give same result: True


---
## Summary

This notebook demonstrated the complete atmospheric calculation capabilities in evtol-tools:

### The Altitude Class

The `Altitude` class is a semantic wrapper around `Length` with aviation-specific features:
- **Creation**: `Altitude(5000, 'ft')`, `Altitude.sea_level()`, `Altitude.from_flight_level(100)`
- **Flight Levels**: `altitude.to_flight_level()` converts to FL number
- **Pressure/Density Altitude**: `Altitude.from_pressure()`, `Altitude.from_density()`
- **Aviation Checks**: `is_in_troposphere()`, `is_below_evtol_ceiling()`
- **Constants**: `TROPOPAUSE_M`, `TROPOPAUSE_FT`, `EVTOL_TYPICAL_CEILING_FT`

### Key Concepts

1. **ISA Model**: Standard atmosphere properties at any altitude (0-80km)
2. **Temperature Offsets**: Model hot day (ISA+) and cold day (ISA-) conditions
3. **Pressure vs Geometric Altitude**: Pressure depends only on geometric altitude
4. **Density Altitude**: The altitude at which ISA matches actual density

### Integration with Propulsion

All propulsion methods accept an `atmosphere` parameter:
- `hover_power_ideal(thrust, atmosphere=...)`
- `hover_shaft_power(thrust, atmosphere=...)`
- `hover_electrical_power(thrust, atmosphere=...)`
- `induced_velocity(thrust, atmosphere=...)`
- `max_tip_speed(atmosphere=...)`

The `atmosphere` parameter accepts:
- `Altitude` object (preferred for clarity)
- `Atmosphere` object for full control
- `Length` object for backwards compatibility

### Key Insights for eVTOL Design

1. **Hover power increases ~15-20% from sea level to 5000 ft**
2. **Hot days (ISA+20K) add another 3-5% power requirement**
3. **Battery sizing must account for worst-case conditions**
4. **Tip Mach limits are more constraining at altitude**
5. **High altitude vertiports require careful performance analysis**
6. **Use `is_below_evtol_ceiling()` to validate operating altitudes**